# Derf Messenger — Android Build (Single Cell)
Run this ONE cell. It installs everything, builds APK (~20-30 min), saves to Drive, downloads.
Before running: upload your 4 files (Derf.py, derf_bg.py, buildozer.spec, buildozer_spec_additions.txt) when the upload dialog appears.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
APK_PATH = '/content/drive/MyDrive/derf.apk'
print('Drive mounted')
import os, sys, time, shutil
os.makedirs('/content/derf', exist_ok=True)
os.chdir('/content/derf')
from google.colab import files
print('Upload: Derf.py, derf_bg.py, buildozer.spec, buildozer_spec_additions.txt')
uploaded = files.upload()
for fn in uploaded: print('Uploaded:', fn, len(open(fn,'rb').read()), 'bytes')
!apt update -qq && apt install -qq -y openjdk-17-jdk-headless ant unzip zip 2>&1 | tail -3
!pip install --upgrade pip setuptools wheel 2>&1 | tail -2
!pip install buildozer==1.6.0 python-for-android==2024.1.21 cython 2>&1 | tail -5
!mkdir -p ~/.android && echo '24333f8a63b6825ea9c5514f83c2829b004d1fee' > ~/.android/repositories.cfg
required = ['Derf.py','derf_bg.py','buildozer.spec','buildozer_spec_additions.txt']
missing = [f for f in required if not os.path.exists(f)]
if missing: raise SystemExit('MISSING FILES: '+str(missing)+'. Upload them before running.')
print('All 4 files verified')
print('=== BUILD STARTED (wait ~20-30 min) ===')
!yes | buildozer -v android debug 2>&1 | tee build.log
import glob
apks = glob.glob('.buildozer/android/**/bin/*.apk', recursive=True)
if apks:
    apk = apks[-1]
    size = os.path.getsize(apk)
    print('APK FOUND:', apk)
    print('Size:', f'{size/1024/1024:.1f} MB')
    shutil.copy(apk, APK_PATH)
    print('Saved to Drive:', APK_PATH)
    files.download(apk)
    print('Download started')
else:
    print('No APK found check build.log')
    !tail -20 build.log